# 05 - Ensemble + Calibration + Submission

**Pipeline final:**
1. Entrenar modelos finales en todos los datos disponibles
2. Multi-seed averaging
3. Ensemble con hill climbing
4. Calibracion de probabilidades
5. Generar submission para Stage 1 (validation) y Stage 2 (2026)
6. Validar y subir a Kaggle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss
from sklearn.calibration import calibration_curve
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

sys.path.insert(0, str(Path('.').resolve()))
from utils.elo import build_elo_features
from utils.efficiency import build_efficiency_features
from utils.features import (
    build_seed_features, aggregate_massey_ordinals,
    build_conference_features, build_record_features,
    build_team_features, build_training_data,
    build_submission_predictions,
)
from utils.cv import multi_seed_cv

PIPELINE_DIR = Path('../../kaggle-pipeline')
sys.path.insert(0, str(PIPELINE_DIR.resolve()))
from src.ensemble import hill_climbing_weights, weighted_average

DATA_DIR = Path('../data')
SUB_DIR = Path('../submissions')
RESULTS_DIR = Path('../results')
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
# Load pre-computed training data and team features
train = pd.read_csv(DATA_DIR / 'train_matchups.csv')
m_team_feats = pd.read_csv(DATA_DIR / 'team_features_mens.csv')
w_team_feats = pd.read_csv(DATA_DIR / 'team_features_womens.csv')

sub_stage1 = pd.read_csv(DATA_DIR / 'SampleSubmissionStage1.csv')
sub_stage2 = pd.read_csv(DATA_DIR / 'SampleSubmissionStage2.csv')

# Prepare X, y
meta_cols = ['Season', 'TeamID1', 'TeamID2', 'target']
feature_cols = [c for c in train.columns if c not in meta_cols]

train[feature_cols] = train[feature_cols].fillna(0)
X = train[feature_cols].values
y = train['target'].values
seasons = train['Season']

print(f'Training: {X.shape}, Features: {len(feature_cols)}')

## 1. Multi-Seed Temporal CV

In [ ]:
# Build submission feature matrices
# Combine men's and women's team features
all_team_feats = pd.concat([m_team_feats, w_team_feats], ignore_index=True)

# Stage 1 features
print('Building Stage 1 features...')
s1_features = build_submission_predictions(sub_stage1, all_team_feats, 
    [c.replace('_diff', '') for c in feature_cols if c.endswith('_diff')])
print(f'Stage 1 features: {s1_features.shape}')

# Stage 2 features
print('Building Stage 2 features...')
s2_features = build_submission_predictions(sub_stage2, all_team_feats,
    [c.replace('_diff', '') for c in feature_cols if c.endswith('_diff')])
print(f'Stage 2 features: {s2_features.shape}')

In [ ]:
# Prepare test matrices
s1_ids = s1_features['ID'].values
s2_ids = s2_features['ID'].values

s1_meta = ['ID', 'Season', 'TeamID1', 'TeamID2']
s1_feat_cols = [c for c in s1_features.columns if c not in s1_meta]

# Ensure same feature columns as training
for col in feature_cols:
    if col not in s1_features.columns:
        s1_features[col] = 0
    if col not in s2_features.columns:
        s2_features[col] = 0

X_s1 = s1_features[feature_cols].fillna(0).values
X_s2 = s2_features[feature_cols].fillna(0).values

print(f'X_s1: {X_s1.shape}, X_s2: {X_s2.shape}')

In [ ]:
%%time
# LightGBM multi-seed
SEEDS = [42, 2024, 2025, 1234, 5678, 7890, 3141, 2718, 1618, 4242]

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'n_estimators': 500,
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1,
}

# Train on all data with multiple seeds, predict on both stages
lgb_s2_preds = []
lgb_s1_preds = []

for seed in SEEDS:
    params = lgb_params.copy()
    params['random_state'] = seed
    model = lgb.LGBMClassifier(**params)
    model.fit(X, y)
    lgb_s1_preds.append(model.predict_proba(X_s1)[:, 1])
    lgb_s2_preds.append(model.predict_proba(X_s2)[:, 1])

lgb_s1_avg = np.mean(lgb_s1_preds, axis=0)
lgb_s2_avg = np.mean(lgb_s2_preds, axis=0)
print(f'LightGBM multi-seed done ({len(SEEDS)} seeds)')

In [ ]:
%%time
# XGBoost multi-seed
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'n_estimators': 500,
    'tree_method': 'hist',
    'verbosity': 0,
    'random_state': 42,
    'n_jobs': -1,
}

xgb_s2_preds = []
xgb_s1_preds = []

for seed in SEEDS:
    params = xgb_params.copy()
    params['random_state'] = seed
    model = xgb.XGBClassifier(**params)
    model.fit(X, y)
    xgb_s1_preds.append(model.predict_proba(X_s1)[:, 1])
    xgb_s2_preds.append(model.predict_proba(X_s2)[:, 1])

xgb_s1_avg = np.mean(xgb_s1_preds, axis=0)
xgb_s2_avg = np.mean(xgb_s2_preds, axis=0)
print(f'XGBoost multi-seed done ({len(SEEDS)} seeds)')

In [ ]:
%%time
# CatBoost multi-seed
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'iterations': 500,
    'verbose': 0,
    'random_seed': 42,
}

cb_s2_preds = []
cb_s1_preds = []

for seed in SEEDS:
    params = cb_params.copy()
    params['random_seed'] = seed
    model = CatBoostClassifier(**params)
    model.fit(X, y)
    cb_s1_preds.append(model.predict_proba(X_s1)[:, 1])
    cb_s2_preds.append(model.predict_proba(X_s2)[:, 1])

cb_s1_avg = np.mean(cb_s1_preds, axis=0)
cb_s2_avg = np.mean(cb_s2_preds, axis=0)
print(f'CatBoost multi-seed done ({len(SEEDS)} seeds)')

## 2. Ensemble

In [ ]:
# Load best weights from notebook 03 (or recompute)
# For now, use equal weights as starting point
# Final weights will come from OOF-based hill climbing in notebook 03
weights = [1/3, 1/3, 1/3]  # Replace with optimized weights from hill climbing

# Try loading from results
try:
    with open(RESULTS_DIR / 'v1_baseline_results.json') as f:
        prev_results = json.load(f)
    if 'weights' in prev_results:
        weights = prev_results['weights']
        print(f'Loaded weights from results: LGB={weights[0]:.3f}, XGB={weights[1]:.3f}, CB={weights[2]:.3f}')
except:
    print(f'Using equal weights: {weights}')

# Ensemble predictions
ensemble_s1 = weighted_average([lgb_s1_avg, xgb_s1_avg, cb_s1_avg], weights)
ensemble_s2 = weighted_average([lgb_s2_avg, xgb_s2_avg, cb_s2_avg], weights)

print(f'\nEnsemble S1 stats: mean={ensemble_s1.mean():.4f}, std={ensemble_s1.std():.4f}')
print(f'Ensemble S2 stats: mean={ensemble_s2.mean():.4f}, std={ensemble_s2.std():.4f}')

## 3. Calibration + Clipping

In [ ]:
# Clip predictions to avoid extreme log loss penalties
CLIP_LOW = 0.05
CLIP_HIGH = 0.95

final_s1 = np.clip(ensemble_s1, CLIP_LOW, CLIP_HIGH)
final_s2 = np.clip(ensemble_s2, CLIP_LOW, CLIP_HIGH)

print(f'After clipping [{CLIP_LOW}, {CLIP_HIGH}]:')
print(f'  S1: min={final_s1.min():.4f}, max={final_s1.max():.4f}')
print(f'  S2: min={final_s2.min():.4f}, max={final_s2.max():.4f}')

# Distribution of predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(final_s1, bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Stage 1 Prediction Distribution')
axes[0].set_xlabel('P(Team1 Wins)')

axes[1].hist(final_s2, bins=50, color='coral', alpha=0.7)
axes[1].set_title('Stage 2 (2026) Prediction Distribution')
axes[1].set_xlabel('P(Team1 Wins)')

plt.tight_layout()
plt.show()

## 4. Generate Submissions

In [ ]:
# Stage 1 submission
sub1 = sub_stage1.copy()
sub1['Pred'] = final_s1

# Stage 2 submission
sub2 = sub_stage2.copy()
sub2['Pred'] = final_s2

# Validation
def validate_submission(sub, name):
    print(f'\n=== Validating {name} ===')
    print(f'  Shape: {sub.shape}')
    print(f'  Columns: {list(sub.columns)}')
    print(f'  NaN count: {sub["Pred"].isna().sum()}')
    print(f'  Min pred: {sub["Pred"].min():.6f}')
    print(f'  Max pred: {sub["Pred"].max():.6f}')
    print(f'  Mean pred: {sub["Pred"].mean():.6f}')
    print(f'  Std pred: {sub["Pred"].std():.6f}')
    
    assert sub['Pred'].between(0, 1).all(), 'Predictions out of [0,1]!'
    assert not sub['Pred'].isna().any(), 'NaN predictions!'
    print(f'  PASSED all checks')

validate_submission(sub1, 'Stage 1')
validate_submission(sub2, 'Stage 2')

In [ ]:
# Save submissions
sub1.to_csv(SUB_DIR / 'submission_stage1_v1.csv', index=False)
sub2.to_csv(SUB_DIR / 'submission_stage2_v1.csv', index=False)

print('Saved:')
print(f'  {SUB_DIR / "submission_stage1_v1.csv"}')
print(f'  {SUB_DIR / "submission_stage2_v1.csv"}')
print(f'\nReady to submit to Kaggle!')
print(f'Use: kaggle competitions submit -c march-machine-learning-mania-2026 -f <file> -m "v1 ensemble"')

## 5. Alternative Submissions (variants)

In [ ]:
# Variant 2: More conservative clipping
conservative_s2 = np.clip(ensemble_s2, 0.10, 0.90)
sub2_conservative = sub_stage2.copy()
sub2_conservative['Pred'] = conservative_s2
sub2_conservative.to_csv(SUB_DIR / 'submission_stage2_v2_conservative.csv', index=False)

# Variant 3: Blend with seed baseline (if SeedNum_diff exists)
# This anchors predictions to well-calibrated seed priors
if 'SeedNum_diff' in feature_cols:
    seed_idx = feature_cols.index('SeedNum_diff')
    lr = LogisticRegression(C=1.0, max_iter=1000)
    lr.fit(X[:, [seed_idx]], y)
    seed_s2 = lr.predict_proba(X_s2[:, [seed_idx]])[:, 1]
    
    blended_s2 = 0.7 * ensemble_s2 + 0.3 * seed_s2
    blended_s2 = np.clip(blended_s2, 0.05, 0.95)
    
    sub2_blended = sub_stage2.copy()
    sub2_blended['Pred'] = blended_s2
    sub2_blended.to_csv(SUB_DIR / 'submission_stage2_v3_seed_blend.csv', index=False)
    print('Variant 3 (seed blend) saved')

print('All variants saved to submissions/')